# 47. The third gate: seven candidates, and one member removed

**One variable against ledger row 94** (`stack_41_all6`, CV 0.968713, public LB 0.97003): the
member set. Same logistic combiner, same `C=1.0`, same logit meta-features, same fold-wise
protocol, same folds.

## Seven in, one out

| candidate | solo CV | rho vs `xgb_te_fe` | what it is |
|---|---|---|---|
| `cat_te_fe` | **0.968036** | 0.991 | row 109, the best single model here |
| `hgb_te_fe` | 0.967885 | 0.996 | row 111, a fourth boosting library |
| `lgb_te_fe` | 0.967212 | - | row 108, positive but inconclusive |
| `rf_te_fe` | 0.962430 | 0.971 | row 113, bagged trees |
| `neural_lookup` | 0.961181 | - | row 107, overfits values, fails differently |
| `et_te_fe` | 0.959538 | 0.972 | row 112, randomised splits |
| **`logit_te_fe`** | **0.954249** | **0.920** | row 110, the first linear model here |

**`lgb_raw` is REMOVED.** It correlates with `bag42` at Pearson 0.999970 on logits because it *is*
`bag42`: both are row 9's configuration, run in two different kernels. Row 77 added it to a stack
that already contained it and the hash quarantine could not see it, because two runs of one
configuration differ by thread-level numerical noise and are not bit-identical. Measured on
2026-08-22: dropping it moves CV by -0.000001. The removal is free and it makes the member count
honest, so the arms below are **47 members, not 48**.

## The check that would have caught it, now in the notebook

Exact-hash quarantine is not enough. This notebook adds a **correlation screen**: any pair above
0.9999 is printed and must be justified. It is three lines and it would have fired the moment row
77 ran.

## What row 94 taught, and why the singles still run

Row 77's five candidates were superadditive, worth 2.9x the sum of their parts. Row 94's six were
**subadditive**, worth 0.39x, because four of them carried the same new information and competed
for the same weight. Neither can be assumed in advance, and running the singles alongside the set
is the only way to know which regime you are in.

This batch is deliberately mixed. `cat_te_fe`, `hgb_te_fe` and `lgb_te_fe` are strong and highly
correlated with what is already there. `logit_te_fe`, `et_te_fe` and `rf_te_fe` are weak and
genuinely decorrelated. If the combiner wants accuracy the first group wins; if it wants
disagreement the second does.

## The prediction, written before the run

**The set clears the floor.** Beyond that: I expect `logit_te_fe` to take a **larger coefficient
than `hgb_te_fe`** despite being 0.0136 weaker, because 0.920 against 0.996 is the widest
disagreement gap this stack has ever been offered.

I am not predicting a magnitude. Four of my last five magnitude predictions were wrong, and the
one honest regularity I have measured about my own reasoning is that my case-against sections
beat my predictions. So here is the case against: the last gate was subadditive, three of these
seven are near-copies of members already present, and a 47-member combiner fitted on five folds
has a lot of freedom. The gain could easily be smaller than row 94's +0.000604.

In [1]:
# 37_stack_views.ipynb
# Membership gate for the five vectors produced by notebooks 35 and 36.
# Runs locally: every member vector already lives in artifacts/oof.
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(b for b in [Path.cwd(), *Path.cwd().parents]
            if (b / "data" / "raw" / "train.csv").exists())
O = ROOT / "artifacts" / "oof"
S = ROOT / "submissions"

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy(np.int8)

# The fold vector is rebuilt rather than loaded, and then checked. A silently different
# fold vector is the one error here that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
FOLD_SHA = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
assert FOLD_SHA == "ec282b0968059676", FOLD_SHA
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")
print(f"fold sha {FOLD_SHA}  VERIFIED")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]
fold sha ec282b0968059676  VERIFIED


In [2]:
# Row 94's forty-one minus the duplicate, in row 94's order, then the five candidates.
BASE = [
    ("te42", "te_bag42"), ("te2024", "te_seed2024"), ("te7", "te_seed7"),
    ("te2025", "te_seed2025"), ("te13", "te_seed13"),
    ("anchor", "lgbm_default_anchor_seed42"), ("trees300", "lgbm_trees300_seed42"),
    ("trees1000", "lgbm_trees1000_seed42"), ("trees2000", "lgbm_trees2000_seed42"),
    ("lr010", "lgbm_lr01_n1000_seed42"), ("lr005", "lgbm_lr005_n2000_seed42"),
    ("lr003", "lgbm_lr003_n3333_seed42"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13"),
    ("neural", "neural"), ("cat42", "catboost_te"), ("cat2024", "catboost_te_seed2024"),
    ("cat7", "catboost_te_seed7"), ("cat2025", "catboost_te_seed2025"),
    ("cat13", "catboost_te_seed13"), ("neural_te", "neural_te"),
    ("xgb_te", "xgb_te"), ("xgb2024", "xgb_te_seed2024"), ("xgb7", "xgb_te_seed7"),
    ("xgb2025", "xgb_te_seed2025"), ("xgb13", "xgb_te_seed13"),
    ("pair_top9", "xgb_pair_top9"),
]
BASE += [("cat_nat_c1", "cat_native_c1"), ("cat_nat_c2", "cat_native_c2"),
         ("xgb_raw", "xgb_raw"), ("cat_raw", "cat_raw"),
         ("xgb_te_fe", "xgb_te_fe"), ("cat_te_n4000", "cat_te_n4000"),
         ("xgb_raw_fe", "xgb_raw_fe"), ("cat_raw_n10k", "cat_raw_n10000"),
         ("lgb_raw_fe", "lgb_raw_fe"), ("cat_raw_fe", "cat_raw_fe")]
# lgb_raw is deliberately absent: it is bag42 re-run in another kernel, Pearson 0.999970 on
# logits. See the header and the 2026-08-22 entry. Dropping it costs -0.000001.
DROPPED = [("lgb_raw", "duplicate of bag42, Pearson 0.999970")]

CAND = [("cat_te_fe", "cat_te_fe"), ("hgb_te_fe", "hgb_te_fe"),
        ("lgb_te_fe", "lgb_te_fe"), ("rf_te_fe", "rf_te_fe"),
        ("neural_lookup", "neural_lookup"), ("et_te_fe", "et_te_fe"),
        ("logit_te_fe", "logit_te_fe")]


def load(stem, kind):
    # OOF/test vector. Two naming conventions exist in artifacts/oof. The bare
    # "{stem}.npy" form is the OOF side only: the early LightGBM members never had a
    # test .npy written and their test side lives in submissions/. Falling back to the
    # bare name for kind="test" silently returns the OOF vector, which is caught by the
    # length assert below only because train and test differ in length.
    cands = [O / f"{stem}_{kind}.npy"]
    if kind == "oof":
        cands.append(O / f"{stem}.npy")
    for c in cands:
        if c.exists():
            return np.load(c)
    if kind == "test" and (S / f"{stem}.csv").exists():
        df = pd.read_csv(S / f"{stem}.csv")
        # A csv written in a different row order blends perfectly cleanly and is
        # undetectable in the score. Checked rather than assumed.
        assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {stem}"
        return df["addicted_label"].to_numpy()
    raise FileNotFoundError(f"{stem} {kind}")


MEM = BASE + CAND
names = [n for n, _ in MEM]
Poof = {n: load(s, "oof") for n, s in MEM}
Ptest = {n: load(s, "test") for n, s in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    assert np.isfinite(Poof[n]).all() and np.isfinite(Ptest[n]).all(), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

# Exact-duplicate quarantine. A duplicated array silently DOUBLES that model's weight.
# Row 59 found xgb_pair_base bit-identical to xgb_te this way and excluded it.
seen = {}
for n in names:
    h = hashlib.md5(np.ascontiguousarray(Poof[n]).tobytes()).hexdigest()
    assert h not in seen, f"{n} is bit-identical to {seen[h]}"
    seen[h] = n
print(f"{len(names)} vectors loaded, no exact duplicates")

# THE CORRELATION SCREEN. Hash equality cannot see one configuration run in two kernels:
# the arrays differ by thread-level numerical noise. bag42 and lgb_raw sat in row 94 at
# Pearson 0.999970 and no check fired. Three lines, and it would have caught them.
Lz = np.column_stack([np.clip(np.log(np.clip(Poof[n], 1e-9, 1 - 1e-9)
                                     / (1 - np.clip(Poof[n], 1e-9, 1 - 1e-9))), -30, 30)
                      for n in names])
Cm = np.corrcoef(((Lz - Lz.mean(0)) / Lz.std(0)).T)
np.fill_diagonal(Cm, 0.0)
near = [(names[i], names[j], Cm[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
        if abs(Cm[i, j]) > 0.9999]
print(f"pairs above 0.9999: {len(near)}")
for a, b, r in near:
    print(f"  NEAR-DUPLICATE {a} and {b} at {r:+.6f}")
assert not near, "a near-duplicate pair is present, justify it or drop one"
print(f"removed from row 94's set: {[d[0] for d in DROPPED]}")
hi = sorted(((abs(Cm[i, j]), names[i], names[j])
             for i in range(len(names)) for j in range(i + 1, len(names))),
            reverse=True)[:3]
print("most collinear surviving pairs: "
      + ", ".join(f"{a}/{b} {r:.5f}" for r, a, b in hi))

47 vectors loaded, no exact duplicates


pairs above 0.9999: 0
removed from row 94's set: ['lgb_raw']
most collinear surviving pairs: cat42/cat7 0.99907, cat2024/cat2025 0.99907, cat2025/cat13 0.99906


In [3]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
IDX = {n: i for i, n in enumerate(names)}
BASE40 = [IDX[n] for n, _ in BASE]

print("candidate solo CV, and disagreement with the members it most resembles:")
print(f"  {'candidate':12} {'solo CV':>10} {'rho vs xgb_te':>15} {'rho vs cat42':>14}")
for n, _ in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    r1 = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    r2 = pd.Series(Poof[n]).corr(pd.Series(Poof["cat42"]), method="spearman")
    print(f"  {n:12} {cv:10.6f} {r1:15.6f} {r2:14.6f}")

# For scale: how decorrelated are two members that everyone agrees are near-copies?
r_seed = pd.Series(Poof["xgb_te"]).corr(pd.Series(Poof["xgb2024"]), method="spearman")
print(f"\n  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): {r_seed:.6f}")
print("  this repo refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.")
print("  This table is context, not a prediction.")

candidate solo CV, and disagreement with the members it most resembles:
  candidate       solo CV   rho vs xgb_te   rho vs cat42


  cat_te_fe      0.968036        0.989789       0.996377


  hgb_te_fe      0.967885        0.994608       0.986885


  lgb_te_fe      0.967212        0.985479       0.979046


  rf_te_fe       0.962430        0.970556       0.978093


  neural_lookup   0.961181        0.961762       0.956979


  et_te_fe       0.959538        0.972886       0.981821


  logit_te_fe    0.954249        0.922665       0.944988



  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): 0.997344
  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.
  This table is context, not a prediction.


In [4]:
def run(cols):
    # Fold-wise logistic combiner. No weight is ever fitted on a row it is scored on.
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    nit = []
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)], y[tr])
        # A non-converged lbfgs fit reads HIGHER than the truth, so convergence is
        # asserted rather than hoped for. Added 2026-08-21 after the public review
        # flagged it; measured at 38 to 40 iterations, so it has never been close.
        nit.append(int(np.max(clf.n_iter_)))
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    assert max(nit) < 2000, f"combiner did not converge, {nit}"
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf, max(nit)


ARMS = {"40_row94": BASE40}
for n, _ in CAND:
    ARMS[f"41_{n}"] = BASE40 + [IDX[n]]
ARMS["47_all7"] = BASE40 + [IDX[n] for n, _ in CAND]

res = {a: run(cols) for a, cols in ARMS.items()}
per = {a: r[0] for a, r in res.items()}

ROW94_CV = 0.968713
repro = per["40_row94"].mean() - ROW94_CV
print(f"reproduction of row 94: {per['40_row94'].mean():.6f} vs {ROW94_CV:.6f}"
      f"  delta {repro:+.2e}   {'REPRODUCED' if abs(repro) < 1e-4 else 'FAILED'}")
print(f"combiner max n_iter across all arms: {max(r[3] for r in res.values())} of 2000\n")

hdr = " ".join(f"{'fold ' + str(i):>9}" for i in range(5))
print(f"{'arm':14} {hdr} {'mean':>10} {'sd':>9}")
for a in ARMS:
    print(f"{a:14} " + " ".join(f"{v:9.6f}" for v in per[a])
          + f" {per[a].mean():10.6f} {per[a].std():9.6f}")

reproduction of row 94: 0.968712 vs 0.968713  delta -7.86e-07   REPRODUCED
combiner max n_iter across all arms: 70 of 2000

arm               fold 0    fold 1    fold 2    fold 3    fold 4       mean        sd
40_row94        0.968106  0.968838  0.968800  0.969333  0.968484   0.968712  0.000407
41_cat_te_fe    0.968136  0.968867  0.968843  0.969362  0.968497   0.968741  0.000409
41_hgb_te_fe    0.968120  0.968849  0.968843  0.969342  0.968488   0.968728  0.000408
41_lgb_te_fe    0.968030  0.968849  0.968819  0.969339  0.968495   0.968706  0.000432
41_rf_te_fe     0.968109  0.968840  0.968795  0.969337  0.968488   0.968714  0.000407
41_neural_lookup  0.968177  0.968878  0.968904  0.969416  0.968575   0.968790  0.000409
41_et_te_fe     0.968108  0.968838  0.968799  0.969341  0.968492   0.968716  0.000408
41_logit_te_fe  0.968108  0.968836  0.968805  0.969334  0.968485   0.968714  0.000407
47_all7         0.968190  0.968914  0.968964  0.969452  0.968600   0.968824  0.000418


In [5]:
FLOOR_MEAN, FLOOR_FOLDS = 0.00005, 4
base_per = per["40_row94"]

print("Paired against row 94's set minus lgb_raw. The gate is >= +0.00005 mean AND >= 4/5 folds.\n")
print(f"{'arm':14} {'paired mean':>13} {'paired sd':>11} {'folds':>7} {'t(4)':>8}  gate")
gate = {}
for a in ARMS:
    if a == "40_row94":
        continue
    d = per[a] - base_per
    wins = int((d > 0).sum())
    sd = d.std(ddof=1)
    t = d.mean() / (sd / np.sqrt(5)) if sd > 0 else float("inf")
    fired = bool(d.mean() >= FLOOR_MEAN and wins >= FLOOR_FOLDS)
    gate[a] = fired
    print(f"{a:14} {d.mean():+13.6f} {sd:11.6f} {wins:5d}/5 {t:8.2f}"
          f"  {'FIRES' if fired else 'under floor'}")

Paired against row 94's set minus lgb_raw. The gate is >= +0.00005 mean AND >= 4/5 folds.

arm              paired mean   paired sd   folds     t(4)  gate
41_cat_te_fe       +0.000029    0.000010     5/5     6.19  under floor
41_hgb_te_fe       +0.000016    0.000015     5/5     2.36  under floor
41_lgb_te_fe       -0.000006    0.000039     4/5    -0.33  under floor
41_rf_te_fe        +0.000002    0.000004     4/5     1.03  under floor
41_neural_lookup     +0.000078    0.000025     5/5     7.08  FIRES
41_et_te_fe        +0.000003    0.000005     3/5     1.62  under floor
41_logit_te_fe     +0.000001    0.000002     4/5     1.42  under floor
47_all7            +0.000112    0.000035     5/5     7.14  FIRES


In [6]:
# Coefficients of the best arm, which is where the result actually lives. Row 59's
# lesson: a model that is null on its own can still take a large weight, and where that
# weight comes FROM is the thing worth reading.
best = max((a for a in ARMS if a != "40_row94"), key=lambda a: per[a].mean())
print(f"best arm by CV: {best}   {per[best].mean():.6f}\n")

cols_b, cols_0 = ARMS[best], ARMS["40_row94"]
cb = res[best][2].mean(axis=0)
c0 = res["40_row94"][2].mean(axis=0)
base_map = {names[c]: c0[i] for i, c in enumerate(cols_0)}

rows = []
for i, c in enumerate(cols_b):
    n = names[c]
    was = base_map.get(n, float("nan"))
    rows.append({"member": n, "coef": cb[i], "was": was, "shift": cb[i] - was})
tab = pd.DataFrame(rows).sort_values("coef", ascending=False)
print(tab.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

new = set(n for n, _ in CAND) & set(tab.member)
gained = tab[tab.member.isin(new)]["coef"].sum()
lost = -tab[~tab.member.isin(new)]["shift"].sum()
print(f"\nnew members carry {gained:+.4f} in total")
print(f"the existing thirty give up {lost:+.4f} of weight between them")
if abs(gained) > 1e-12:
    print(f"substitution covers {100 * lost / gained:.0f} percent of the new weight")

best arm by CV: 47_all7   0.968824

       member    coef     was   shift
   cat_nat_c2 +0.2981 +0.2973 +0.0009
    cat_te_fe +0.2125     NaN     NaN
   xgb_raw_fe +0.1830 +0.2245 -0.0415
    xgb_te_fe +0.1193 +0.2234 -0.1041
    hgb_te_fe +0.1146     NaN     NaN
   lgb_raw_fe +0.1142 +0.1430 -0.0288
 cat_te_n4000 +0.0914 +0.1828 -0.0913
 cat_raw_n10k +0.0837 +0.1052 -0.0215
    pair_top9 +0.0793 +0.0917 -0.0124
neural_lookup +0.0758     NaN     NaN
   cat_raw_fe +0.0559 +0.1960 -0.1401
    lgb_te_fe +0.0409     NaN     NaN
      cat2024 +0.0384 +0.0736 -0.0353
      xgb2024 +0.0376 +0.0528 -0.0152
        lr003 +0.0350 +0.0394 -0.0043
        lr005 +0.0304 +0.0281 +0.0023
    neural_te +0.0283 +0.0700 -0.0416
         xgb7 +0.0272 +0.0350 -0.0077
       neural +0.0246 +0.0042 +0.0204
        cat13 +0.0230 +0.0490 -0.0260
      cat2025 +0.0165 +0.0419 -0.0253
        xgb13 +0.0145 +0.0263 -0.0118
      xgb_raw +0.0125 -0.0149 +0.0274
      bag2025 +0.0061 +0.0011 +0.0050
    trees1000 

In [7]:
# Three decisions, kept separate. Bundling them was the error corrected in row 59.
print("1. GATE")
for a, f in gate.items():
    print(f"     {a:14} {'FIRES' if f else 'under floor'}")

print("\n2. MEMBERSHIP")
print("   A sub-floor addition is still kept and logged as negligible: row 32 kept four")
print("   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership")
print("   follows the sign and the fold count, not the floor.")
keep = [a for a in ARMS if a != "40_row94"
        and (per[a] - base_per).mean() > 0
        and int(((per[a] - base_per) > 0).sum()) >= 4]
print(f"   arms positive and >= 4/5 folds: {keep if keep else 'none'}")
print(f"   carried forward: {best} at {per[best].mean():.6f}")

print("\n3. SUBMISSION")
SUB = S / "stack_v3.csv"
if per[best].mean() > ROW94_CV:
    p = res[best][1].mean(axis=0)
    sub = pd.DataFrame({"id": test["id"].to_numpy(),
                        "addicted_label": (np.argsort(np.argsort(p)) + 0.5) / len(p)})
    assert len(sub) == len(test) and np.isfinite(sub["addicted_label"]).all()
    sub.to_csv(SUB, index=False)
    print(f"   wrote {SUB.name}, {len(sub):,} rows,"
          f" {sub['addicted_label'].nunique():,} distinct")
    print("   AUC reads order only, so the rank transform changes nothing and keeps the")
    print("   file comparable with the earlier stack submissions.")
else:
    print(f"   no submission: best arm {per[best].mean():.6f} does not beat row 94")

print(f"\nledger lines:\n  name    stack_{best}\n  cv_mean {per[best].mean():.6f}"
      f"\n  cv_std  {per[best].std():.6f}")

1. GATE
     41_cat_te_fe   under floor
     41_hgb_te_fe   under floor
     41_lgb_te_fe   under floor
     41_rf_te_fe    under floor
     41_neural_lookup FIRES
     41_et_te_fe    under floor
     41_logit_te_fe under floor
     47_all7        FIRES

2. MEMBERSHIP
   A sub-floor addition is still kept and logged as negligible: row 32 kept four
   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership
   follows the sign and the fold count, not the floor.
   arms positive and >= 4/5 folds: ['41_cat_te_fe', '41_hgb_te_fe', '41_rf_te_fe', '41_neural_lookup', '41_logit_te_fe', '47_all7']
   carried forward: 47_all7 at 0.968824

3. SUBMISSION


   wrote stack_v3.csv, 296,302 rows, 296,302 distinct
   AUC reads order only, so the rank transform changes nothing and keeps the
   file comparable with the earlier stack submissions.

ledger lines:
  name    stack_47_all7
  cv_mean 0.968824
  cv_std  0.000418
